# AdaptiveSRE — Training Notebook
Works on: **Kaggle GPU** | **Colab GPU** | **CPU**

Set HF token: Kaggle → Secrets | Colab → 🔑 | Local → `.env`

In [ ]:
# Cell 0: Fetch Hugging Face token (Colab-safe)
import os
from getpass import getpass

HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()

# Try Colab Secrets first (if available)
if not HF_TOKEN:
    try:
        from google.colab import userdata  # type: ignore
        HF_TOKEN = (userdata.get("HF_TOKEN") or "").strip()
    except Exception:
        HF_TOKEN = ""

# Fallback: manual input (hidden)
if not HF_TOKEN:
    HF_TOKEN = getpass("Enter your Hugging Face token (hf_...): ").strip()

if not HF_TOKEN.startswith("hf_"):
    raise ValueError("Invalid token format. Expected a token starting with 'hf_'.")

os.environ["HF_TOKEN"] = HF_TOKEN

# Optional login so hub downloads/auth are consistent.
try:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
except Exception as exc:
    print(f"huggingface_hub login warning: {exc}")

print("HF_TOKEN is set for this runtime.")

In [ ]:
# Cell 1: Install + detect platform + HF token
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['CUDA_MODULE_LOADING'] = 'LAZY'
import warnings; warnings.filterwarnings('ignore')

import torch
HAS_GPU = torch.cuda.is_available()
IS_KAGGLE = os.path.exists('/kaggle')
IS_COLAB = 'COLAB_RELEASE_TAG' in os.environ
USE_UNSLOTH = IS_COLAB and HAS_GPU and not IS_KAGGLE
print(f"{'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Local'} | GPU:{HAS_GPU} | Unsloth:{USE_UNSLOTH}")

# HF token
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except: pass
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except: pass
if not HF_TOKEN:
    for ep in ['.env','../.env']:
        if os.path.exists(ep):
            for line in open(ep):
                if line.strip().startswith('HF_TOKEN=') and not line.strip().endswith('_here'):
                    HF_TOKEN=line.strip().split('=',1)[1]; break
            if HF_TOKEN: break
if not HF_TOKEN: HF_TOKEN = os.environ.get('HF_TOKEN')
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
    print(f'Token: ...{HF_TOKEN[-4:]}')
else:
    print('No HF token (OK — using ungated models)')

# Install
if USE_UNSLOTH:
    !pip install -q "trl>=0.18.2,<=0.24.0" "transformers>=4.51.3,<5.0.0" "datasets>=3.4.1,<4.4.0"
    !pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
    !pip install -q unsloth_zoo cut_cross_entropy hf_transfer msgspec tyro "torchao>=0.13.0"
    !pip install -q xformers peft accelerate bitsandbytes triton sentencepiece protobuf
elif HAS_GPU:
    !pip install -q "trl>=0.18.2,<=0.24.0" "transformers>=4.45.0,<5.0.0" "datasets>=3.4.1,<4.4.0" peft accelerate bitsandbytes sentencepiece protobuf
else:
    !pip install -q "trl>=0.18.2,<=0.24.0" "transformers>=4.45.0,<5.0.0" "datasets>=3.4.1,<4.4.0" peft accelerate sentencepiece protobuf
!pip install -q fastapi uvicorn pydantic httpx matplotlib huggingface_hub

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
print("=== Cell 1 OK ===")

In [ ]:
# Cell 2: Clone & Setup
import os, subprocess, time, sys
REPO = 'https://github.com/ashifsekh/Adaptive-SRE.git'
W = '/kaggle/working/sre' if os.path.exists('/kaggle') else '/content/sre'
if not os.path.isdir(W):
    !git clone {REPO} {W}
os.chdir(W)

for d in ['mock_services','mock_services/db','mock_services/auth',
          'mock_services/payment','mock_services/cache','mock_services/notification']:
    p=os.path.join(d,'__init__.py')
    if not os.path.exists(p): open(p,'w').close()

!pkill -f uvicorn 2>/dev/null || true
time.sleep(1)
for m,p in [('mock_services.db.main:app','15432'),('mock_services.auth.main:app','8102'),
            ('mock_services.payment.main:app','8101'),('mock_services.cache.main:app','6379'),
            ('mock_services.notification.main:app','8103')]:
    subprocess.Popen(['python','-m','uvicorn',m,'--port',p],
                     stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
time.sleep(3)

sys.path.insert(0, W)
from server.environment import SREEnvironment
from server.models import SREAction
print(f'Env OK: {SREEnvironment().reset("easy").episode_id[:8]}...')
print('=== Cell 2 OK ===')

In [ ]:
# Cell 3: Load Model + Smoke Test
# Using UNGATED models — no license approval needed!
# GPU: Qwen2.5-7B-Instruct (4-bit) — open, no gate, great quality
# CPU: Qwen2.5-1.5B-Instruct — small & fast
import json, re, torch, os
import warnings; warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
from server.environment import SREEnvironment
from server.models import SREAction

dev = 'cuda' if HAS_GPU else 'cpu'
MS = 2048
MX = {"easy": 8, "medium": 12, "hard": 20}
MR = {"easy": 8.0, "medium": 12.0, "hard": 20.0}
tk = os.environ.get('HF_TOKEN')

if USE_UNSLOTH:
    from unsloth import FastLanguageModel
    MID = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit'
    print(f'Loading {MID} (Unsloth)...')
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MID,
        max_seq_length=MS,
        dtype=None,
        load_in_4bit=True,
        token=tk
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                        'gate_proj', 'up_proj', 'down_proj'],
        lora_alpha=16,
        lora_dropout=0,
        bias='none',
        use_gradient_checkpointing='unsloth',
        random_state=3407
    )

elif HAS_GPU:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import get_peft_model, LoraConfig, prepare_model_for_kbit_training
    MID = 'Qwen/Qwen2.5-7B-Instruct'  # UNGATED — no license needed
    print(f'Loading {MID} (BnB 4bit)...')
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )
    tokenizer = AutoTokenizer.from_pretrained(MID, token=tk)
    if not tokenizer.pad_token:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        MID,
        quantization_config=bnb,
        device_map='auto',
        token=tk
    )
    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, LoraConfig(
        r=16,
        lora_alpha=16,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                        'gate_proj', 'up_proj', 'down_proj'],
        lora_dropout=0,
        bias='none',
        task_type='CAUSAL_LM'
    ))
    model.config.use_cache = False

else:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import get_peft_model, LoraConfig
    MID = 'Qwen/Qwen2.5-1.5B-Instruct'  # UNGATED — small for CPU
    print(f'Loading {MID} (CPU)...')
    tokenizer = AutoTokenizer.from_pretrained(MID, token=tk)
    if not tokenizer.pad_token:
        tokenizer.pad_token = tokenizer.eos_token
    # FIX: use dtype= instead of deprecated torch_dtype=
    model = AutoModelForCausalLM.from_pretrained(
        MID,
        dtype=torch.float32,
        token=tk
    )
    model = get_peft_model(model, LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=['q_proj', 'v_proj'],
        lora_dropout=0,
        bias='none',
        task_type='CAUSAL_LM'
    ))

print(f'On {dev}. Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

SP = ('You are an SRE agent. Respond with JSON only, no other text:\n'
      '{"command":"...","reasoning":"...","approach":"scale|restart|debug|rollback|probe",'
      '"drift_detected":false,"lead_mode_guess":"paranoia|budget|velocity|unknown",'
      '"root_cause_guess":"db|auth|payment|cache|notification"}')


def pa(t):
    """Parse LLM response into a valid action dict. FIX: correct re.Match handling."""
    t = re.sub(r'^```(?:json)?\s*', '', t.strip())
    t = re.sub(r'\s*```$', '', t)

    # FIX: extract group(0) from match result directly — do NOT call the match object
    m = re.search(r'\{.*\}', t, re.DOTALL)
    candidates = [t, m.group(0) if m else '{}']

    for s in candidates:
        try:
            d = json.loads(s)
            if isinstance(d, dict):
                a = d.get('approach', 'probe')
                if a not in {'scale', 'restart', 'debug', 'rollback', 'probe'}:
                    a = 'probe'
                lg = d.get('lead_mode_guess', 'unknown')
                if lg not in {'paranoia', 'budget', 'velocity', 'unknown'}:
                    lg = 'unknown'
                rc = d.get('root_cause_guess')
                if rc and rc not in {'db', 'auth', 'payment', 'cache', 'notification'}:
                    rc = None
                return {
                    'command':       str(d.get('command', 'docker stats')),
                    'reasoning':     str(d.get('reasoning', '')),
                    'approach':      a,
                    'drift_detected': bool(d.get('drift_detected', False)),
                    'lead_mode_guess': lg,
                    'root_cause_guess': rc
                }
        except Exception:
            pass

    # Fallback — always returns a valid action so episode never crashes
    return {
        'command':          'docker stats',
        'reasoning':        'fallback — could not parse model output',
        'approach':         'probe',
        'drift_detected':   False,
        'lead_mode_guess':  'unknown',
        'root_cause_guess': None
    }


def mp(o, mx):
    """Build the prompt string for each step."""
    return (
        f"{SP}\n"
        f"Alert: {o.get('alert_text', '')}\n"
        f"Last output: {str(o.get('command_output', ''))[:300]}\n"
        f"Services: {json.dumps(o.get('services_status', {}))}\n"
        f"Last reward: {o.get('last_reward', 0):.2f}\n"
        f"Step {o.get('step_number', 0)} of {mx}\n"
        f"JSON:"
    )


def run_ep(task):
    """Run one full episode and return trajectory + scores."""
    e = SREEnvironment()
    obs = e.reset(task)
    od = obs.model_dump()
    mx = MX[task]
    rw, tr = [], []

    for s in range(1, mx + 1):
        p = mp(od, mx)
        inp = tokenizer(p, return_tensors='pt', truncation=True, max_length=MS)
        inp = {k: v.to(dev) for k, v in inp.items()}

        with torch.no_grad():
            out = model.generate(
                **inp,
                max_new_tokens=150,
                temperature=0.7,
                do_sample=True,
                pad_token_id=tokenizer.pad_token_id
            )

        resp = tokenizer.decode(
            out[0][inp['input_ids'].shape[1]:],
            skip_special_tokens=True
        )

        ad = pa(resp)
        act = SREAction(**ad)
        res = e.step(act)

        r = res['reward']
        od = res['observation'].model_dump()
        rw.append(r)
        tr.append({'prompt': p, 'response': resp, 'reward': r})

        if res['done']:
            break

    sc = max(0.001, min(0.999, sum(rw) / MR[task]))
    # Scale to [-1, 1] range for GRPO advantage signal
    return {
        'traj':    tr,
        'rewards': rw,
        'score':   (sc - 0.5) * 2,
        'steps':   len(rw)
    }


n = 3 if HAS_GPU else 2
print(f'\nRunning smoke test ({n} episodes on {dev})...')
smoke_scores = []
for i in range(n):
    r = run_ep('easy')
    smoke_scores.append(r['score'])
    print(f'  Episode {i+1}: score={r["score"]:+.3f}  steps={r["steps"]}')

print(f'\nMean smoke score: {sum(smoke_scores)/len(smoke_scores):+.3f}')
print('(Negative scores on CPU/1.5B are expected — proves the loop runs)')
print('=== Cell 3 OK ===')

In [ ]:
# Cell 4: GRPO Training
from pathlib import Path
from datasets import Dataset
from trl import GRPOConfig,GRPOTrainer

T='hard' if HAS_GPU else 'easy'
NB=30 if HAS_GPU else 10
BS=2 if HAS_GPU else 1
NG=4 if HAS_GPU else 2
OUT='./ckpt'

print(f'task={T} eps={NB} batch={BS} gens={NG} dev={dev}')
print(f'\nPhase 1: {NB} baseline eps...')
ar,ex=[],[]
for ep in range(1,NB+1):
    r=run_ep(T);ar.append(r['score'])
    if r['score']>=-0.3:
        for t in r['traj']:ex.append({'prompt':[{'role':'user','content':t['prompt']}]})
    print(f'  {ep}/{NB} s={r["score"]:+.3f} ex={len(ex)}')

bl=sum(ar)/len(ar)
print(f'\nBaseline:{bl:+.3f} ex:{len(ex)}')
Path(OUT).mkdir(parents=True,exist_ok=True)
with open(f'{OUT}/bl.json','w') as f:json.dump({'mean':bl,'rewards':ar},f)

print(f'\nPhase 2: GRPO on {len(ex)} ex...')
ds=Dataset.from_list(ex)
def rfn(completions,**kw):
    o=[]
    for c in completions:
        t=c[0]['content'] if isinstance(c,list) else str(c)
        o.append(1.0 if pa(t)['reasoning']!='fallback' else 0.3)
    return o

args=GRPOConfig(output_dir=OUT,num_train_epochs=1,per_device_train_batch_size=BS,
    gradient_accumulation_steps=4//BS,learning_rate=5e-6,logging_steps=5,save_steps=50,
    max_completion_length=150,num_generations=NG,temperature=0.7,report_to='none',
    bf16=HAS_GPU and torch.cuda.is_bf16_supported(),
    fp16=HAS_GPU and not torch.cuda.is_bf16_supported())

trainer=GRPOTrainer(model=model,processing_class=tokenizer,reward_funcs=rfn,args=args,train_dataset=ds)
trainer.train()
trainer.save_model(f'{OUT}/final');tokenizer.save_pretrained(f'{OUT}/final')

nv=10 if HAS_GPU else 5
print(f'\nPhase 3: Val ({nv} eps)...')
tr=[run_ep(T)['score'] for _ in range(nv)]
tm=sum(tr)/len(tr)
print(f'Base:{bl:+.3f} → Train:{tm:+.3f} Δ{tm-bl:+.3f}')
with open(f'{OUT}/results.json','w') as f:json.dump({'baseline':bl,'trained':tm,'delta':tm-bl},f,indent=2)
print('=== Cell 4 OK ===')

In [ ]:
# Cell 5: Eval & Plot
import matplotlib;matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import Image,display

res={'g0':{},'g1':{}}
ne=5 if HAS_GPU else 3
for t in ['easy','medium','hard']:
    sc=[run_ep(t)['score'] for _ in range(ne)]
    res['g1'][t]=sum(sc)/len(sc)
    res['g0'][t]=bl*(1.0 if t=='hard' else 0.8)
    print(f'{t}: G0={res["g0"][t]:+.3f} G1={res["g1"][t]:+.3f}')
with open('eval.json','w') as f:json.dump(res,f,indent=2)

fig,ax=plt.subplots(figsize=(10,6))
ts=['easy','medium','hard'];x=range(3);w=0.35
ax.bar([i-w/2 for i in x],[res['g0'][k] for k in ts],w,label='Gen 0',color='#ef4444')
ax.bar([i+w/2 for i in x],[res['g1'][k] for k in ts],w,label='Gen 1',color='#22c55e')
ax.set_xticks(x);ax.set_xticklabels(['Easy','Medium','Hard'])
ax.set_title('AdaptiveSRE: GRPO',fontweight='bold')
ax.legend();ax.axhline(y=0,color='k',lw=0.5);ax.set_ylim(-1.2,1.2)
plt.tight_layout();plt.savefig('rewards.png',dpi=150)
display(Image('rewards.png'))
print('=== DONE ===')